# Lab 4.2 · A plan within the budget

Day 4 · Session 2 (Choosing under a budget) · 25 min.

Lab 4.1's best actions would spend more than Juhaina's capital budget. You rank the actions by expected saving per riyal invested, fill the budget, and report the plan as Nouran needs it: what it costs, how many bursts it avoids with an interval, who carries the remaining risk, and which districts the model knows least about.

**AI policy.** The tutor Mirsad may explain; it writes no code for the plan.

**Reference state.** `data/posterior/rates.npz` and `data/costs.json`; the costs of Lab 4.1 are recomputed below, so Lab 4.1 need not be finished.

**Tasks**
- [ ] 1 · The plan without a budget, and its cost (4')
- [ ] 2 · Benefit per riyal (5')
- [ ] 3 · Fill the budget (6')
- [ ] 4 · Avoided bursts with an interval (6')
- [ ] 5 · Who carries the remaining risk (4')
- [ ] ✱ Stretch · Solve the budget exactly by dynamic programming and compare with the greedy plan

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO_URL = "https://github.com/SDAIAAcademy/ds-401.git"
    if not Path("/content/ds-401").exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/ds-401"], check=True)
    os.chdir("/content/ds-401/labs")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
else:
    while not Path("yanabi.py").exists() and Path.cwd() != Path.cwd().parent:
        os.chdir("..")  # solutions/ notebooks run from labs/
sys.path.insert(0, os.getcwd())

import numpy as np
import pandas as pd
import yanabi as U
print("mode:", U.MODE)

ACTIONS = ["wait", "repair", "replace"]
rate, ids = U.posterior_rates()
c = U.costs()
dist_tbl = U.districts().set_index("district_id").loc[ids]
km = dist_tbl["network_km"].to_numpy()
H, B = c["horizon_years"], c["cost_per_burst"]
invest = {a: c["actions"][a]["cost_per_km"] * km for a in ACTIONS}
avoided_per_draw = {a: rate * km * c["actions"][a]["years"] * c["actions"][a]["reduction"] for a in ACTIONS}  # bursts avoided
saving = {a: (avoided_per_draw[a] * B).mean(axis=0) - invest[a] for a in ACTIONS}  # expected net saving against waiting
BUDGET = c["capital_budget"]
print(f"capital budget: SAR {BUDGET / 1e6:.0f} million")

## 1 · The plan without a budget, and its cost (4')

✏️ **Exercise** · For each district take the action with the largest expected net saving (waiting saves 0). Return the total investment of that plan and the number of districts per action.

In [ ]:
def unconstrained():
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")

In [ ]:
try:
    best_u, spend_u = unconstrained()
except NotImplementedError:
    best_u, spend_u = (np.array(v) if i == 0 else v for i, v in enumerate(U.reference("d4_2", "unconstrained")))
U.keep("d4_2", "unconstrained", [list(best_u), spend_u])
print(pd.Series(best_u).value_counts().to_string())
print(f"investment without a budget: SAR {spend_u / 1e6:.1f} million against a budget of {BUDGET / 1e6:.0f}")
U.check(spend_u > BUDGET, "the unconstrained plan does not fit the budget")

## 2 · Benefit per riyal (5')

Rank the steps of investment by expected net saving per riyal. A district can take two steps: from waiting to repair, then from repair to replacement (the extra cost buys the extra saving). Where replacement is worth more per riyal than repair, the district's step is straight from waiting to replacement.

✏️ **Exercise** · Return a DataFrame of steps with `district_id`, `step` (`repair`, `replace` or `repair->replace`), `invest` and `saving` (both increments) and `ratio`, keeping only steps with a positive saving, sorted by `ratio`, largest first.

In [ ]:
def options():
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")

In [ ]:
try:
    opts = options()
except NotImplementedError:
    opts = U.reference("d4_2", "options")
U.keep("d4_2", "options", opts)
print(opts.head(8).round(2).to_string(index=False))
U.check(len(opts) > 0 and opts["ratio"].is_monotonic_decreasing, "steps sorted by saving per riyal")

## 3 · Fill the budget (6')

Go down the list and take each step that fits in what is left of the budget; an upgrade to replacement only for a district already given repair. (A greedy rule: fast and close to the best, not always the best; the stretch checks it.)

✏️ **Exercise** · Return the plan as a dict `district_id -> action` (`repair` or `replace`), its total investment and its total expected net saving.

In [ ]:
def fill_budget():
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")

In [ ]:
try:
    plan, spent, saved = fill_budget()
except NotImplementedError:
    plan, spent, saved = U.reference("d4_2", "plan")
U.keep("d4_2", "plan", [plan, spent, saved])
print(pd.Series(plan).value_counts().to_string())
print(f"plan: {len(plan)} districts, SAR {spent / 1e6:.1f} million invested, expected net saving SAR {saved / 1e6:.1f} million")
U.check(spent <= BUDGET and spent > 0.9 * BUDGET, "the plan uses most of the budget without exceeding it")

## 4 · Avoided bursts with an interval (6')

Bursts avoided over the horizon differ from draw to draw: the plan's uncertainty is the posterior's.

✏️ **Exercise** · Return, per draw, the total bursts the plan avoids, and print its median and 90% interval.

In [ ]:
def avoided():
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")

In [ ]:
try:
    av = avoided()
except NotImplementedError:
    av = np.array(U.reference("d4_2", "avoided"))
U.keep("d4_2", "avoided", av[:200])
lo, md_, hi = np.quantile(av, [0.05, 0.5, 0.95])
print(f"bursts avoided over {H} years: {md_:.0f} (90% interval {lo:.0f} to {hi:.0f})")
U.check(hi > lo * 1.1, "the avoided bursts carry real uncertainty")

## 5 · Who carries the remaining risk (4')

✏️ **Exercise** · Return two tables: the 8 unfunded districts with the most expected bursts over the horizon (who carries the remaining risk), and the 8 districts with the widest posterior relative to their median (what the model knows least about), each with kind, sensors and whether the plan funds them.

In [ ]:
def remaining_risk():
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")

In [ ]:
try:
    unfunded, unknown = remaining_risk()
except NotImplementedError:
    unfunded, unknown = U.reference("d4_2", "unfunded"), U.reference("d4_2", "unknown")
U.keep("d4_2", "unfunded", unfunded)
U.keep("d4_2", "unknown", unknown)
print("unfunded, most expected bursts:\n" + unfunded.round(2).to_string())
print("\nleast known:\n" + unknown.round(2).to_string())
U.check((~unknown["sensors"]).mean() >= 0.5, "most of the least-known districts have no sensors")

**Look at** the two tables. The unfunded urban districts carry most of the bursts the plan leaves; their residents and the repair crews carry that risk. The least-known districts are mostly rural without sensors: the plan's choices for them rest on the audit prior, which is what Lab 4.3's survey would sharpen.

## ✱ Stretch · The exact optimum

Solve the same problem exactly: a dynamic programme over budget units of SAR 4,000 (every investment is a multiple of it), where each district chooses wait, repair or replace. How much more does the exact plan save than the greedy one?

In [ ]:
def stretch():
    # ✏️ YOUR CODE HERE
    raise NotImplementedError("✏️ Exercise: write your solution here")


try:
    stretch()
except NotImplementedError:
    print("✱ stretch not done")

In [ ]:
U.note("Checked: the plan fits the budget and carries its uncertainty in avoided bursts.",
       "Named: who carries the remaining risk, and where the model knows least.",
       "Carry forward: the plan; Lab 4.3 asks whether better data on the least-known districts is worth its price.")

## Checkpoint

1. Why rank by saving per riyal rather than by saving?
   <details><summary>Answer</summary>The budget is the constraint: a district that saves less but costs far less can give more saving per riyal spent.</details>
2. Why report avoided bursts with an interval?
   <details><summary>Answer</summary>The rates are uncertain, so the plan's effect is too; a single number hides how wrong it could be.</details>
3. Who carries the remaining risk?
   <details><summary>Answer</summary>The residents and crews of the districts the plan leaves, named in the memo with their expected bursts.</details>

## Carry forward

The memo of the capstone reports the same four things: the plan, its cost, its expected effect with an interval, and the remaining risk.